In [35]:
import polars as pl
import pandas as pd
import json 
from comorbidipy import comorbidity, ScoreType, hfrs
import icu_flag
import hosps
import cim10_utils
import comorbidity_utils
import drug
import patient_utils

In [36]:
unit_mapping = pl.read_parquet("unit_mapping.parquet")
stay = pl.read_parquet("stay.parquet")
document_data = pl.read_parquet("document_data.parquet")
treatments = pl.read_parquet("traitement.parquet")
patient = pl.read_parquet("patient.parquet")

In [37]:
unit_mapping_icu = icu_flag.add_icu_unit_flag(unit_mapping)

In [38]:
stay_with_icu = icu_flag.add_icu_flag_to_stay(stay, unit_mapping_icu)

In [39]:
hosp = hosps.get_hosp(stay_with_icu)

In [40]:
inclusion_hosps = hosps.get_inclusion_hosps(hosp)

In [41]:
first_inclusion_hosps = hosps.get_first_inclusion_hosps(inclusion_hosps)

In [42]:
preceding_hosps = hosps.get_preceding_hosps(hosp)

In [43]:
included_patients = hosps.get_included_patients(first_inclusion_hosps)

In [44]:
preceding_hosps_of_included_patients = hosps.get_preceding_hosps_of_included_patients(preceding_hosps, included_patients)

In [45]:
document_data = document_data.with_columns(
    pl.when(pl.col("terminology_code") == "CIM-10")
    .then(pl.lit("cim10"))
    .otherwise(pl.col("terminology_code"))
    .alias("terminology_code")
)

In [46]:
cim10 = cim10_utils.load_cim10(document_data)

In [47]:
preceding_hosps_of_included_patients_with_cim10 = cim10_utils.get_preceding_hosps_of_included_patients_with_cim10(preceding_hosps_of_included_patients, cim10)

In [48]:
disease_dict = comorbidity_utils.load_disease_dict('cim10_nancy.json')

In [49]:
comorbidities = comorbidity_utils.get_comorbidities(
    preceding_hosps_of_included_patients_with_cim10,
    disease_dict
)

In [50]:
patient_comorbidities = comorbidity_utils.get_patient_comorbidities(comorbidities)

In [51]:
disease_cols = ["Cardiopathie ischémique", "Fibrillation atriale", "Insuffisance cardiaque chronique",
               "Pacemaker", "Pontage aorto-coronarien", "Insuffisance rénale chronique", "Antécédent d'AVC",
               "Cirrhose", "Cancer actif", "Immunodépression", "Dyslipidémie", "Diabète"]


In [52]:
included_patients_with_comorbidity =comorbidity_utils.get_included_patients_with_comorbidity(
    first_inclusion_hosps,
    patient_comorbidities,
    disease_cols
)

In [53]:
treatments_with_flags = drug.create_treatment_flags(treatments)

In [54]:
patient_treatment_flags = drug.get_patient_treatment_flags(treatments_with_flags)

In [55]:
drug_cols = ["beta_blocker", "ACEI", "ARB",  "anti_aldosterone",  "furosemide"]

In [56]:
included_patient_with_comorbidity_drug = drug.get_patient_with_comorbidity_drug(
    included_patients_with_comorbidity,
    patient_treatment_flags,
    drug_cols
)

In [57]:
patients_with_death_date = patient_utils.get_patients_with_death_date(patient)

In [58]:
included_patient_with_comorbidity_drug_demo = included_patient_with_comorbidity_drug.join(patients_with_death_date, how = "left", on = "patient_id")

In [59]:
included_patient_with_comorbidity_drug_demo = patient_utils.age_at_admission(included_patient_with_comorbidity_drug_demo)

In [60]:
data_for_score = (
    preceding_hosps_of_included_patients_with_cim10
    .select(["patient_id", "concept_code"])
    .filter(pl.col("concept_code").is_not_null())
    .join(
        included_patient_with_comorbidity_drug_demo
        .select(["patient_id", "age_at_admission"])
        .unique(),
        how="left",
        on="patient_id"
    )
    .unique()
)
#data_for_score_pd = data_for_score.to_pandas()

In [61]:
data_for_score

patient_id,concept_code,age_at_admission
str,str,i32
"""P001""","""I48""",85
"""P003""","""N18""",83
"""P008""","""I693""",80
"""P010""","""I48""",83
"""P005""","""E785""",81
"""P004""","""I21""",82
"""P001""","""I25""",85
"""P002""","""E119""",84
"""P005""","""C349""",81


In [62]:
charlson_score = comorbidity(
    
    data_for_score,
        
    id_col="patient_id",
        
    code_col="concept_code",
        
    score=ScoreType.CHARLSON,
)

hfrs_score = hfrs(
    data_for_score, 
    id_col="patient_id",
    code_col="concept_code")

In [63]:
charlson_selected = charlson_score.select([
    "patient_id",
    "comorbidity_score"
])

hfrs_selected = hfrs_score.select([
    "patient_id",
    "hfrs_score"
])

final_patient_data = (
    included_patient_with_comorbidity_drug_demo
    .join(charlson_selected, on="patient_id", how="left")
    .join(hfrs_selected, on="patient_id", how="left")
    .with_columns([
        pl.col("comorbidity_score").fill_null(0),
        pl.col("hfrs_score").fill_null(0),
    ])
)

In [64]:
final_patient_data = final_patient_data.with_columns([
    (
        pl.col("beta_blocker").fill_null(0)
        + pl.max_horizontal([
            pl.col("ACEI").fill_null(0),
            pl.col("ARB").fill_null(0),
            #pl.col("ARNI").fill_null(0),
        ])
        + pl.col("anti_aldosterone").fill_null(0)
        #+ pl.col("sglt2i").fill_null(0)
    ).alias("OMT_component_score")
])

In [65]:
# Creat death_flag
final_patient_data = final_patient_data.with_columns([
    pl.when(pl.col("death_date").is_not_null())
    .then(1)
    .otherwise(0)
    .alias("death_flag")
])


patient_id,hosp_id,centre,hosp_admission_datetime,hosp_discharge_datetime,inclusion_flag,icu_stay_flag,Cardiopathie ischémique,Fibrillation atriale,Insuffisance cardiaque chronique,Pacemaker,Pontage aorto-coronarien,Insuffisance rénale chronique,Antécédent d'AVC,Cirrhose,Cancer actif,Immunodépression,Dyslipidémie,Diabète,beta_blocker,ACEI,ARB,anti_aldosterone,furosemide,gender,birth_date,death_date,age_at_admission,comorbidity_score,hfrs_score,OMT_component_score,death_flag
str,i64,str,datetime[μs],datetime[μs],i64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,str,date,date,i32,f64,f64,i8,i32
"""P001""",1002,"""angers""",2018-07-04 11:15:00,2018-07-12 10:00:00,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,"""F""",1933-04-12,2018-09-21,85,0.0,0.0,2,1
"""P002""",2002,"""brest""",2019-01-18 13:20:00,2019-01-25 11:00:00,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""M""",1934-11-03,null,84,0.0,0.0,0,0
"""P003""",3002,"""nantes""",2019-04-09 09:15:00,2019-04-18 10:30:00,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,1,1,"""F""",1935-07-26,2020-12-04,83,3.0,1.4,2,1
"""P004""",4002,"""tours""",2018-03-03 07:50:00,2018-03-18 12:20:00,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,"""M""",1936-02-18,null,82,0.0,0.0,1,0
"""P005""",5002,"""angers""",2019-02-01 14:10:00,2019-02-08 15:00:00,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,"""F""",1937-09-09,2019-03-17,81,2.0,0.0,0,1
"""P006""",6002,"""brest""",2020-06-17 10:45:00,2020-06-24 12:00:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""M""",1938-12-22,null,81,0.0,0.0,0,0
"""P007""",7001,"""nantes""",2020-07-22 08:30:00,2020-07-29 10:15:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""F""",1939-05-15,null,81,0.0,0.0,0,0
"""P008""",8002,"""tours""",2020-12-05 06:45:00,2020-12-20 16:00:00,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,"""M""",1940-01-30,2021-07-02,80,0.0,3.7,0,1
"""P009""",9002,"""angers""",2017-01-26 09:00:00,2017-02-03 11:45:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""F""",1933-10-05,null,83,0.0,0.0,0,0


In [67]:
final_patient_data.write_parquet("final_patient_data.parquet")